In [42]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

print(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from src.config import (
    TRAIN_PATH,
    TEST_PATH,
    SAMPLE_SUBMISSION_PATH,
    SUBMISSION_DIR,
    TARGET,
    ID_COL,
    RANDOM_STATE,
    N_SPLITS
)

c:\Users\pc\Desktop\yzta-2026-datathon


In [43]:
def add_features(df):
    df = df.copy()

    # Uyku kalitesi
    if {"rem_yuzdesi", "derin_uyku_yuzdesi"}.issubset(df.columns):
        df["toplam_kaliteli_uyku_yuzdesi"] = (
            df["rem_yuzdesi"] + df["derin_uyku_yuzdesi"]
        )

        df["rem_derin_uyku_carpim"] = (
            df["rem_yuzdesi"] * df["derin_uyku_yuzdesi"]
        )

    # Uyku bölünmesi
    if {"gecelik_uyanma_sayisi", "uykuya_dalma_suresi_dk"}.issubset(df.columns):
        df["uyku_bolunme_yuku"] = (
            df["gecelik_uyanma_sayisi"] * df["uykuya_dalma_suresi_dk"]
        )

        df["uyku_verimsizlik_skoru"] = (
            df["uykuya_dalma_suresi_dk"] + 10 * df["gecelik_uyanma_sayisi"]
        )

    # Stres ve çalışma yükü
    if {"stres_skoru", "gunluk_calisma_saati"}.issubset(df.columns):
        df["stres_calisma_yuku"] = (
            df["stres_skoru"] * df["gunluk_calisma_saati"]
        )

    # Ekran + kafein
    if {"uyku_oncesi_ekran_suresi_dk", "uyku_oncesi_kafein_mg"}.issubset(df.columns):
        df["ekran_kafein_yuku"] = (
            df["uyku_oncesi_ekran_suresi_dk"] + df["uyku_oncesi_kafein_mg"]
        )

    # Adım sayısı
    if "gunluk_adim_sayisi" in df.columns:
        df["adim_sayisi_bin"] = df["gunluk_adim_sayisi"] / 1000

    # Nabız + stres
    if {"dinlenik_nabiz_bpm", "stres_skoru"}.issubset(df.columns):
        df["nabiz_stres_yuku"] = (
            df["dinlenik_nabiz_bpm"] * df["stres_skoru"]
        )

    # BMI kategorisi
    if "vucut_kitle_indeksi" in df.columns:
        df["bmi_kategori"] = pd.cut(
            df["vucut_kitle_indeksi"],
            bins=[0, 18.5, 25, 30, np.inf],
            labels=["zayif", "normal", "kilolu", "obez"]
        ).astype("object")

    # Hafta sonu flag
    if "gun_tipi" in df.columns:
        df["hafta_sonu_flag"] = (df["gun_tipi"] == "Hafta sonu").astype(int)

    # Ruh sağlığı risk skoru
    if "ruh_sagligi_durumu" in df.columns:
        risk_map = {
            "Saglikli": 0,
            "Anksiyete": 1,
            "Depresyon": 2,
            "Anksiyete ve depresyon": 3,
        }

        df["ruh_sagligi_risk_skoru"] = df["ruh_sagligi_durumu"].map(risk_map)

    return df

In [44]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

Train shape: (56000, 24)
Test shape: (24000, 23)
Sample submission shape: (2, 2)


In [45]:
train_fe = add_features(train)
test_fe = add_features(test)

print("Train FE shape:", train_fe.shape)
print("Test FE shape:", test_fe.shape)

Train FE shape: (56000, 35)
Test FE shape: (24000, 34)


In [46]:
X = train_fe.drop(columns=[TARGET, ID_COL])
y = train_fe[TARGET]

X_test = test_fe.drop(columns=[ID_COL])
test_ids = test_fe[ID_COL]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (56000, 33)
y shape: (56000,)
X_test shape: (24000, 33)


In [47]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric feature count:", len(numeric_features))
print("Categorical feature count:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

Numeric feature count: 25
Categorical feature count: 8

Categorical features:
['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi', 'bmi_kategori']


In [48]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

In [49]:
def rmse(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return np.sqrt(mse)


cv = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [50]:
models = {
    "catboost": CatBoostRegressor(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        random_seed=RANDOM_STATE,
        verbose=0
    ),

    "lightgbm": LGBMRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=-1,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "xgboost": XGBRegressor(
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [51]:
results = []
oof_predictions = {}
test_predictions = {}

for model_name, model in models.items():
    print("=" * 80)
    print(f"Model: {model_name}")

    oof_pred = np.zeros(len(X))
    test_pred_folds = np.zeros((len(X_test), N_SPLITS))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
        print(f"Fold {fold}")

        X_train_fold = X.iloc[train_idx]
        X_valid_fold = X.iloc[valid_idx]

        y_train_fold = y.iloc[train_idx]
        y_valid_fold = y.iloc[valid_idx]

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train_fold, y_train_fold)

        valid_pred = pipeline.predict(X_valid_fold)
        valid_pred = np.clip(valid_pred, 0, 10)

        fold_rmse = rmse(y_valid_fold, valid_pred)
        fold_scores.append(fold_rmse)

        oof_pred[valid_idx] = valid_pred

        test_pred = pipeline.predict(X_test)
        test_pred = np.clip(test_pred, 0, 10)
        test_pred_folds[:, fold - 1] = test_pred

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    mean_rmse = np.mean(fold_scores)
    std_rmse = np.std(fold_scores)

    oof_predictions[model_name] = oof_pred
    test_predictions[model_name] = test_pred_folds.mean(axis=1)

    results.append({
        "model": model_name,
        "cv_rmse_mean": mean_rmse,
        "cv_rmse_std": std_rmse,
        "fold_scores": fold_scores
    })

    print(f"{model_name} CV RMSE: {mean_rmse:.5f} ± {std_rmse:.5f}")

Model: catboost
Fold 1
Fold 1 RMSE: 1.22419
Fold 2
Fold 2 RMSE: 1.22403
Fold 3
Fold 3 RMSE: 1.20821
Fold 4
Fold 4 RMSE: 1.21970
Fold 5
Fold 5 RMSE: 1.23827
catboost CV RMSE: 1.22288 ± 0.00965
Model: lightgbm
Fold 1
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001528 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4611
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 72
[LightGBM] [Info] Start training from score 5.913723


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 RMSE: 1.23984
Fold 2
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002051 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4610
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 72
[LightGBM] [Info] Start training from score 5.909518


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 RMSE: 1.24165
Fold 3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001782 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4609
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 72
[LightGBM] [Info] Start training from score 5.913069


...

Fold 3 RMSE: 1.22716
Fold 4
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001434 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4615
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 72
[LightGBM] [Info] Start training from score 5.917491


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 RMSE: 1.23586
Fold 5
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001964 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4610
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 72
[LightGBM] [Info] Start training from score 5.911677


c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
c:\Users\pc\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 RMSE: 1.25576
lightgbm CV RMSE: 1.24005 ± 0.00931
Model: xgboost
Fold 1
Fold 1 RMSE: 1.24178
Fold 2
Fold 2 RMSE: 1.24173
Fold 3
Fold 3 RMSE: 1.22645
Fold 4
Fold 4 RMSE: 1.24013
Fold 5
Fold 5 RMSE: 1.25726
xgboost CV RMSE: 1.24147 ± 0.00977


In [52]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("cv_rmse_mean").reset_index(drop=True)

results_df[["model", "cv_rmse_mean", "cv_rmse_std"]]

,model,cv_rmse_mean,cv_rmse_std
0,catboost,1.222878,0.009646
1,lightgbm,1.240055,0.009308
2,xgboost,1.241470,0.009766


In [53]:
catboost_oof_rmse = rmse(y, oof_predictions["catboost"])

print("CatBoost OOF RMSE:", catboost_oof_rmse)

CatBoost OOF RMSE: 1.222916413387271


In [54]:
ensemble_oof = (
    0.40 * oof_predictions["catboost"] +
    0.35 * oof_predictions["lightgbm"] +
    0.25 * oof_predictions["xgboost"]
)

ensemble_oof = np.clip(ensemble_oof, 0, 10)

ensemble_rmse = rmse(y, ensemble_oof)

print("Ensemble OOF RMSE:", ensemble_rmse)

Ensemble OOF RMSE: 1.227395424077371


In [55]:
ensemble_oof = (
    0.70 * oof_predictions["catboost"] +
    0.15 * oof_predictions["lightgbm"] +
    0.15 * oof_predictions["xgboost"]
)

ensemble_oof = np.clip(ensemble_oof, 0, 10)

ensemble_rmse = rmse(y, ensemble_oof)

print("CatBoost RMSE:", rmse(y, oof_predictions["catboost"]))
print("Ensemble RMSE:", ensemble_rmse)

CatBoost RMSE: 1.222916413387271
Ensemble RMSE: 1.2240049417744703


In [56]:
ensemble_results = []

weights_list = [
    (0.80, 0.10, 0.10),
    (0.70, 0.15, 0.15),
    (0.60, 0.20, 0.20),
    (0.50, 0.25, 0.25),
    (0.60, 0.30, 0.10),
    (0.60, 0.10, 0.30),
    (0.75, 0.20, 0.05),
    (0.75, 0.05, 0.20),
]

for w_cat, w_lgb, w_xgb in weights_list:
    ensemble_oof = (
        w_cat * oof_predictions["catboost"] +
        w_lgb * oof_predictions["lightgbm"] +
        w_xgb * oof_predictions["xgboost"]
    )

    ensemble_oof = np.clip(ensemble_oof, 0, 10)
    score = rmse(y, ensemble_oof)

    ensemble_results.append({
        "catboost_weight": w_cat,
        "lightgbm_weight": w_lgb,
        "xgboost_weight": w_xgb,
        "rmse": score
    })

ensemble_results_df = pd.DataFrame(ensemble_results)
ensemble_results_df = ensemble_results_df.sort_values("rmse").reset_index(drop=True)

ensemble_results_df

,catboost_weight,lightgbm_weight,xgboost_weight,rmse
0,0.80,0.10,0.10,1.223375
1,0.75,0.20,0.05,1.223442
2,0.70,0.15,0.15,1.224005
3,0.75,0.05,0.20,1.224096
4,0.60,0.30,0.10,1.224725
5,0.60,0.20,0.20,1.224902
6,0.60,0.10,0.30,1.225479
7,0.50,0.25,0.25,1.226066


In [57]:
catboost_test_pred = test_predictions["catboost"]
catboost_test_pred = np.clip(catboost_test_pred, 0, 10)

submission_catboost = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: catboost_test_pred
})

submission_catboost.head()

,id,bilissel_performans_skoru
0,1,5.969426
1,2,6.433410
2,3,2.866489
3,4,7.163522
4,5,3.669634


In [58]:
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

catboost_submission_path = SUBMISSION_DIR / "submission_catboost.csv"

submission_catboost.to_csv(catboost_submission_path, index=False)

print("Saved:", catboost_submission_path)

Saved: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_catboost.csv


In [59]:
print("Submission shape:", submission_catboost.shape)
print("Sample submission shape:", sample_submission.shape)

print("Submission columns:", submission_catboost.columns.tolist())
print("Sample columns:", sample_submission.columns.tolist())

display(submission_catboost.head())
display(sample_submission.head())

Submission shape: (24000, 2)
Sample submission shape: (2, 2)
Submission columns: ['id', 'bilissel_performans_skoru']
Sample columns: ['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.969426
1,2,6.433410
2,3,2.866489
3,4,7.163522
4,5,3.669634


,id,bilissel_performans_skoru
0,1,7.85
1,2,4.32


In [60]:
ensemble_test_pred = (
    0.40 * test_predictions["catboost"] +
    0.35 * test_predictions["lightgbm"] +
    0.25 * test_predictions["xgboost"]
)

ensemble_test_pred = np.clip(ensemble_test_pred, 0, 10)

print("Prediction min:", ensemble_test_pred.min())
print("Prediction max:", ensemble_test_pred.max())
print("Prediction mean:", ensemble_test_pred.mean())

Prediction min: 0.024816840155328996
Prediction max: 10.0
Prediction mean: 5.93941959433981


In [61]:
submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: ensemble_test_pred
})

submission.head()

,id,bilissel_performans_skoru
0,1,5.853687
1,2,6.450332
2,3,2.850360
3,4,7.152588
4,5,3.778539


In [62]:
print("Submission shape:", submission.shape)
print("Sample submission shape:", sample_submission.shape)

print("\nSubmission columns:")
print(submission.columns.tolist())

print("\nSample submission columns:")
print(sample_submission.columns.tolist())

display(submission.head())
display(sample_submission.head())

Submission shape: (24000, 2)
Sample submission shape: (2, 2)

Submission columns:
['id', 'bilissel_performans_skoru']

Sample submission columns:
['id', 'bilissel_performans_skoru']


,id,bilissel_performans_skoru
0,1,5.853687
1,2,6.450332
2,3,2.850360
3,4,7.152588
4,5,3.778539


,id,bilissel_performans_skoru
0,1,7.85
1,2,4.32


In [63]:
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

submission_path = SUBMISSION_DIR / "submission_final_ensemble.csv"

submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

Saved submission to: C:\Users\pc\Desktop\yzta-2026-datathon\submissions\submission_final_ensemble.csv
